In [57]:
import os
import zipfile
import urllib.request
import pandas as pd
import numpy as np


# Directory & path setup

In [58]:
DATA_DIR = os.path.join(".", "data", "raw")
ZIP_PATH = os.path.join(DATA_DIR, "bank_marketing.zip")
TARGET_CSV = os.path.join(DATA_DIR, "bank-full.csv")

# UCI Bank Marketing Dataset URL

In [59]:
UCI_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
os.makedirs(DATA_DIR, exist_ok=True)

In [60]:
# 1. Download if not already present
if not os.path.exists(TARGET_CSV):
    if not os.path.exists(ZIP_PATH):
        print("[*] Downloading dataset from UCI...")
        urllib.request.urlretrieve(UCI_URL, ZIP_PATH)
        print(f"[+] Download complete: {ZIP_PATH}")
    
    # 2. Extract outer zip
    print(f"[*] Extracting files to {DATA_DIR}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as outer_zip:
        outer_zip.extractall(DATA_DIR)
        
    # 3. Extract inner bank.zip (UCI stores bank.zip inside bank+marketing.zip)
    inner_zip_path = os.path.join(DATA_DIR, "bank.zip")
    if os.path.exists(inner_zip_path):
        with zipfile.ZipFile(inner_zip_path, 'r') as inner_zip:
            inner_zip.extractall(DATA_DIR)

In [61]:
df = pd.read_csv(TARGET_CSV, sep=';')
print(f"[+] Dataset successfully loaded: {TARGET_CSV}")
print(f"    - Total instances: {df.shape[0]:,}")
print(f"    - Total attributes: {df.shape[1]}")
print("\nTarget Class Distribution ('y'):")
print(df['y'].value_counts())
print("\nClass Proportion (%):")
print((df['y'].value_counts(normalize=True) * 100).round(2).astype(str) + '%')
df.head(5)

[+] Dataset successfully loaded: .\data\raw\bank-full.csv
    - Total instances: 45,211
    - Total attributes: 17

Target Class Distribution ('y'):
y
no     39922
yes     5289
Name: count, dtype: int64

Class Proportion (%):
y
no     88.3%
yes    11.7%
Name: proportion, dtype: str


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [62]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

In [63]:
data = df.copy()
y = (data['y'] == 'yes').astype(int)

 Prevent Target Leakage: Drop 'duration' and target 'y'
# In realistic telemarketing, call duration is unknown prior to dialing.

In [64]:
X_raw = data.drop(columns=['y', 'duration'])

Handle 'pdays' Sentinel Value (-1 means 'not previously contacted')
# - Create a clean binary flag: 'previously_contacted' (1 if contacted, 0 otherwise)
# - For 'log_pdays', log-transform positive days so -1 becomes 0 instead of distorting numeric scale

In [65]:
X_raw['previously_contacted'] = (X_raw['pdays'] != -1).astype(int)
pdays_clean = np.where(X_raw['pdays'] > 0, X_raw['pdays'], 0)
X_raw['log_pdays'] = np.log1p(pdays_clean)
X_raw = X_raw.drop(columns=['pdays'])
print("[+] Cleaned successfully! Current columns in X_raw:")
print(X_raw.columns.tolist())

[+] Cleaned successfully! Current columns in X_raw:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'campaign', 'previous', 'poutcome', 'previously_contacted', 'log_pdays']


In [66]:
numeric_cols = ['age', 'balance', 'day', 'campaign', 'previous', 'log_pdays']
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

Preprocessor:
# - Categorical: One-Hot Encode (keeps 'unknown' as its own column, drops 1st to prevent collinearity)
# - Numeric: StandardScaler (zero mean, unit variance)

In [67]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_cols),
        ('passthrough', 'passthrough', ['previously_contacted'])
    ]
)

X_processed = preprocessor.fit_transform(X_raw)

In [68]:
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
feature_names = numeric_cols + list(cat_feature_names) + ['previously_contacted']
X = pd.DataFrame(X_processed, columns=feature_names)

print("[+] Preprocessing Complete!")
print(f"    - Original raw shape: {df.shape}")
print(f"    - Clean Feature Matrix (X) shape: {X.shape}")
print(f"    - Target Vector (y) length: {len(y)} (Positives: {y.sum():,}, Negatives: {(y==0).sum():,})")
print(f"    - Total Features Created: {len(feature_names)}")
X.head(3)

[+] Preprocessing Complete!
    - Original raw shape: (45211, 17)
    - Clean Feature Matrix (X) shape: (45211, 42)
    - Target Vector (y) length: 45211 (Positives: 5,289, Negatives: 39,922)
    - Total Features Created: 42


,age,balance,day,campaign,previous,log_pdays,job_blue-collar,job_entrepreneur,job_housemaid,job_management,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown,previously_contacted
0,1.606965,0.256419,-1.298476,-0.569351,-0.25194,-0.467314,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.288529,-0.437895,-1.298476,-0.569351,-0.25194,-0.467314,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.747384,-0.446762,-1.298476,-0.569351,-0.25194,-0.467314,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [70]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix, precision_recall_curve
)
import warnings
warnings.filterwarnings('ignore')

In [72]:
RATIO_CONFIGS = {
    "1:20 (Severe, 4.8%)": {"pos_ratio": 1/21, "n_pos": 1900, "n_neg": 38000},
    "1:10 (High, 9.1%)":   {"pos_ratio": 1/11, "n_pos": 3800, "n_neg": 38000},
    "1:7.5 (Natural, 11.7%)": {"pos_ratio": 5289/45211, "n_pos": 5289, "n_neg": 39922},
    "1:3 (Moderate, 25.0%)":  {"pos_ratio": 1/4,  "n_pos": 5289, "n_neg": 15867},
    "1:1 (Balanced, 50.0%)":  {"pos_ratio": 1/2,  "n_pos": 5289, "n_neg": 5289}
}

SEEDS = [42, 101, 777, 999, 2024]

In [73]:
def get_models():
    return {
        "Logistic Regression": LogisticRegression(C=1.0, max_iter=1000, random_state=42),
        "Decision Tree": DecisionTreeClassifier(max_depth=7, min_samples_leaf=25, min_samples_split=50, random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=10, random_state=42, n_jobs=-1)
    }

In [75]:
results_records = []
pr_curves_data = {}  # for plotting curves in the next step
# Separate indices by class
pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]

print(f"[*] Starting controlled benchmark across {len(RATIO_CONFIGS)} ratios and {len(SEEDS)} seeds...")
for ratio_name, cfg in RATIO_CONFIGS.items():
    print(f"\n--> Running Ratio: {ratio_name} (Pos: {cfg['n_pos']:,}, Neg: {cfg['n_neg']:,})")
    pr_curves_data[ratio_name] = {}


[*] Starting controlled benchmark across 5 ratios and 5 seeds...

--> Running Ratio: 1:20 (Severe, 4.8%) (Pos: 1,900, Neg: 38,000)

--> Running Ratio: 1:10 (High, 9.1%) (Pos: 3,800, Neg: 38,000)

--> Running Ratio: 1:7.5 (Natural, 11.7%) (Pos: 5,289, Neg: 39,922)

--> Running Ratio: 1:3 (Moderate, 25.0%) (Pos: 5,289, Neg: 15,867)

--> Running Ratio: 1:1 (Balanced, 50.0%) (Pos: 5,289, Neg: 5,289)


In [77]:
for ratio_name, cfg in RATIO_CONFIGS.items():
    print(f"\n--> Running Ratio: {ratio_name} (Pos: {cfg['n_pos']:,}, Neg: {cfg['n_neg']:,})")
    pr_curves_data[ratio_name] = {}
    
    for seed in SEEDS:
        rng = np.random.RandomState(seed)
        
        # Subsample to controlled ratio
        selected_pos = rng.choice(pos_idx, size=cfg['n_pos'], replace=False)
        selected_neg = rng.choice(neg_idx, size=cfg['n_neg'], replace=False)
        sub_indices = np.concatenate([selected_pos, selected_neg])
        rng.shuffle(sub_indices)
        
        X_sub = X.iloc[sub_indices]
        y_sub = y.iloc[sub_indices]
        
        # Stratified 80/20 train/test split (prevalence preserved in test set)
        X_train, X_test, y_train, y_test = train_test_split(
            X_sub, y_sub, test_size=0.20, stratify=y_sub, random_state=seed
        )
        models = get_models()
        for model_name, model in models.items():
            model.fit(X_train, y_train)
            
            # Predictions & Continuous Probabilities
            y_pred = model.predict(X_test)
            y_prob = model.predict_proba(X_test)[:, 1]
            
            # Confusion Matrix components
            cm = confusion_matrix(y_test, y_pred)
            tn, fp, fn, tp = cm.ravel()
            
            # Metrics
            acc = accuracy_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred, zero_division=0)
            rec = recall_score(y_test, y_pred, zero_division=0)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            auc_pr = average_precision_score(y_test, y_prob)
            
            results_records.append({
                "Ratio": ratio_name,
                "Seed": seed,
                "Model": model_name,
                "Accuracy": acc,
                "Precision": prec,
                "Recall": rec,
                "F1_Score": f1,
                "AUC_PR": auc_pr,
                "TP": tp, "FP": fp, "TN": tn, "FN": fn
            })


--> Running Ratio: 1:20 (Severe, 4.8%) (Pos: 1,900, Neg: 38,000)

--> Running Ratio: 1:10 (High, 9.1%) (Pos: 3,800, Neg: 38,000)

--> Running Ratio: 1:7.5 (Natural, 11.7%) (Pos: 5,289, Neg: 39,922)

--> Running Ratio: 1:3 (Moderate, 25.0%) (Pos: 5,289, Neg: 15,867)

--> Running Ratio: 1:1 (Balanced, 50.0%) (Pos: 5,289, Neg: 5,289)


In [78]:
if seed == SEEDS[0]:
                prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_prob)
                pr_curves_data[ratio_name][model_name] = {
                    "precision": prec_curve,
                    "recall": rec_curve,
                    "auc_pr": auc_pr,
                    "prevalence": cfg['pos_ratio']
                }

In [79]:
df_results = pd.DataFrame(results_records)
summary_table = df_results.groupby(["Ratio", "Model"])[
    ["Accuracy", "Precision", "Recall", "F1_Score", "AUC_PR"]
].agg(['mean', 'std']).round(4)
print("\n" + "="*80)
print("[+] ALL EXPERIMENTS COMPLETED!")
print("="*80)
summary_table


[+] ALL EXPERIMENTS COMPLETED!


Accuracy         Precision          \
                                               mean     std      mean     std   
Ratio                  Model                                                    
1:1 (Balanced, 50.0%)  Decision Tree         0.6997  0.0083    0.7580  0.0201   
                       Logistic Regression   0.7000  0.0110    0.7377  0.0197   
                       Random Forest         0.7242  0.0111    0.7731  0.0203   
1:10 (High, 9.1%)      Decision Tree         0.9139  0.0008    0.5907  0.0203   
                       Logistic Regression   0.9139  0.0011    0.6126  0.0256   
                       Random Forest         0.9158  0.0007    0.6665  0.0126   
1:20 (Severe, 4.8%)    Decision Tree         0.9514  0.0007    0.4485  0.0417   
                       Logistic Regression   0.9531  0.0010    0.5593  0.0822   
                       Random Forest         0.9524  0.0002    0.5489  0.3812   
1:3 (Moderate, 25.0%)  Decision Tree         0.7992  0.0096    0.7156  0.0291   
                       Logistic Regression   0.8014  0.0068    0.7303  0.0227   
                       Random Forest         0.8026  0.0072    0.7179  0.0227   
1:7.5 (Natural, 11.7%) Decision Tree         0.8913  0.0019    0.6260  0.0495   
                       Logistic Regression   0.8925  0.0015    0.6492  0.0291   
                       Random Forest         0.8942  0.0018    0.7082  0.0390   

                                            Recall         F1_Score          \
                                              mean     std     mean     std   
Ratio                  Model                                                  
1:1 (Balanced, 50.0%)  Decision Tree        0.5881  0.0276   0.6618  0.0135   
                       Logistic Regression  0.6216  0.0124   0.6745  0.0098   
                       Random Forest        0.6355  0.0177   0.6973  0.0115   
1:10 (High, 9.1%)      Decision Tree        0.1742  0.0154   0.2686  0.0177   
                       Logistic Regression  0.1455  0.0096   0.2351  0.0136   
                       Random Forest        0.1479  0.0086   0.2420  0.0121   
1:20 (Severe, 4.8%)    Decision Tree        0.0895  0.0170   0.1488  0.0251   
                       Logistic Regression  0.0800  0.0063   0.1398  0.0109   
                       Random Forest        0.0068  0.0051   0.0134  0.0100   
1:3 (Moderate, 25.0%)  Decision Tree        0.3261  0.0431   0.4469  0.0455   
                       Logistic Regression  0.3251  0.0220   0.4498  0.0254   
                       Random Forest        0.3459  0.0243   0.4667  0.0265   
1:7.5 (Natural, 11.7%) Decision Tree        0.1820  0.0164   0.2811  0.0164   
                       Logistic Regression  0.1781  0.0033   0.2794  0.0055   
                       Random Forest        0.1631  0.0072   0.2651  0.0113   

                                            AUC_PR          
                                              mean     std  
Ratio                  Model                                
1:1 (Balanced, 50.0%)  Decision Tree        0.7521  0.0146  
                       Logistic Regression  0.7818  0.0102  
                       Random Forest        0.8005  0.0101  
1:10 (High, 9.1%)      Decision Tree        0.2918  0.0087  
                       Logistic Regression  0.3536  0.0094  
                       Random Forest        0.3783  0.0081  
1:20 (Severe, 4.8%)    Decision Tree        0.1761  0.0172  
                       Logistic Regression  0.2326  0.0216  
                       Random Forest        0.2451  0.0195  
1:3 (Moderate, 25.0%)  Decision Tree        0.5031  0.0135  
                       Logistic Regression  0.5903  0.0180  
                       Random Forest        0.6125  0.0212  
1:7.5 (Natural, 11.7%) Decision Tree        0.3250  0.0104  
                       Logistic Regression  0.4040  0.0067  
                       Random Forest        0.4325  0.0075